In [1]:
import requests 
from bs4 import BeautifulSoup as bs
import pandas as pd
import tldextract
import numpy as np
import matplotlib.pylab as plt
import os
import re

In [2]:
def on_sale_chk(text):
    if len(text)<1:
        return False
    return 'domain' in text and 'sale' in text

def on_parked_chk(text):
    if len(text)<1:
        return True
    return 'domain' in text and 'park' in text

def on_Parked(text):
    if len(text)<1:
        return True
    return (('website' in text or 'content' in text) and 'unavailable' in text) or ('will' in text and 'soon' in text)

In [3]:
#returns html contents, textual character length, website size, status code, parked or on sale

def soupFromUrl(scrapeUrl):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows; U; Windows NT 6.1; zh-CN) AppleWebKit/533+ (KHTML, like Gecko)'}
    try:
        req = requests.get(scrapeUrl, headers=headers, timeout=5)
        # print(req.status_code)
        req.close()
        if req.status_code == 200:
            # print(bs(req.text, 'html.parser').get_text().strip().replace('\n',' '))
            soup = bs(req.text,'html')

            # print(soup)

            text = ''

            if soup.body:
                text = re.sub(r'[^\w]', ' ',soup.body.get_text(' ', strip=True).lower())

            # print(soup)

            # print('text',text)
            # print(bs(req.text, 'html.parser'))
            # return [bs(req.text, 'html.parser'),len(req.text), len(req.content), req.status_code]
            # print([len(req.content), len(text), req.status_code, 0+(on_sale_chk(text) or on_Parked(text) or on_parked_chk(text))])
            return [len(req.content), len(text), req.status_code, 0+(on_sale_chk(text) or on_Parked(text) or on_parked_chk(text))]
        else:
            # return [-1,0,0,req.status_code]
            return [0,0,req.status_code,0]
    except:
        # return [-1,0,0,-1]
        return [0,0,-1,0]

In [4]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows; U; Windows NT 6.1; zh-CN) AppleWebKit/533+ (KHTML, like Gecko)'}
req = requests.get('https://www.lycos.com/', headers=headers, timeout=5)
print(req.status_code)
req.close()
if req.status_code == 200:
    # print(bs(req.text, 'html.parser').get_text().strip().replace('\n',' '))
    soup = bs(req.text,'html')

    # print(soup)
    text = ''

    if soup.body:
        text = re.sub(r'[^\w]', ' ',soup.body.get_text(' ', strip=True).lower())

    print(soup)
    print('text',text)
    print([len(req.content), len(text), req.status_code, 0+(on_sale_chk(text) or on_Parked(text) or on_parked_chk(text))])

200
<!DOCTYPE html>

<html lang="en">
<head>
<meta charset="utf-8"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<!-- The above 3 meta tags *must* come first in the head; any other head content must come *after* these tags -->
<meta content="Lycos, Inc., is a web search engine and web portal established in 1994, spun out of Carnegie Mellon University. Lycos also encompasses a network of email, webhosting, social networking, and entertainment websites." name="description"/>
<meta content="" name="author"/>
<link href="https://ly.lygo.net/static/lycos/img/favicon.ico" rel="icon" type="image/png"/>
<title>Lycos.com</title>
<link href="//fonts.googleapis.com/css?family=Lato:400,300,300italic,400italic,700,700italic" rel="stylesheet" type="text/css"/>
<link href="/css/in/fonts.css" rel="stylesheet" type="text/css">
<link href="https://ly.lygo.net/static/lycos/css/in/font-awesome.css" rel="stylesheet" type="text

In [5]:
# print(soupFromUrl('https://www.delinian.com/'))
# print(soupFromUrl('https://www.makecashonline.com/'))
print(soupFromUrl('https://www.lycos.com/'))

[13852, 328, 200, 0]


In [7]:
spamhunter_url = list(pd.read_csv('../Dataset/URL Data/Spam-Hunter.csv',delimiter='\t')['URL'])
spamhunter_url[:10]

['bit.ly/BantuanDanaBPJS',
 'http://www.ethinfo.net',
 'bit.ly/2JTIngsungATM',
 'googl.com/?coin',
 'post.com/?=es2938456',
 'https://bit.ly/36900m8',
 'http://umu.link/raboscanner',
 'bit.ly/tiktok-7947',
 'http://rewardspayout.com/',
 'cy9.co/tpo']

In [8]:
import re

def find_first_slash_preceded_by_number(s):
    # Regular expression to find the first instance of a number followed by '/'
    match = re.search(r'\d+/', s)
    
    if match:
        return match.start() + len(match.group()) - 1  # Return the index of '/'
    else:
        return -1  # Return -1 if no match is found

# Example usage
string = "example77/test 88/test2 99/test3"
index = find_first_slash_preceded_by_number(string)
print(index)  # Outputs the index of the first '/' preceded by a number

9


In [13]:
unique_spamhunter_url = list(set(spamhunter_url))

In [14]:
for idx,i in enumerate(unique_spamhunter_url):
    if '..' in i:
        if 'www' in i:
            unique_spamhunter_url[idx] = ''
        else:
            unique_spamhunter_url[idx] = unique_spamhunter_url[idx].replace('..','.')

In [25]:
def refine_url(url):
    # Trim invalid leading characters (keep only letters, numbers, or valid schemes)
    trimmed_url = re.sub(r"^[^a-zA-Z0-9]+", "", url)
    
    # Trim invalid trailing characters (must end with a letter, digit, `/`, `?`, `#`, or `=`)
    trimmed_url = re.sub(r"[^a-zA-Z0-9/?#=]+$", "", trimmed_url)

    # Ensure URL starts correctly (with a scheme or valid domain character)
    if not re.match(r"^(https?://|ftp://|[a-zA-Z0-9])", trimmed_url):
        return ""  # Invalid start, return empty

    return trimmed_url


unique_spamhunter_url = [refine_url(i) for i in unique_spamhunter_url]
print(unique_spamhunter_url)

['http://www.kycpaytm.com/in.php', 'points.co.in', 'bit.ly/im3', 'https://bit.ly/3goHo9H', 'privee.com', '3.uk/18e', 'argops.com', 'http://www.karamozyar.ir/dhl/?10iztoui77wo', 'http://c.p9tj.com', '33700.fr', 'https://paypal.co.uk.yn5e.eu/m/', 'bit.ly/', 's.id/8080spor7', 'amb.web.id', 'bit.ly/anular-recibos', 'bit.ly/39fUjHk', 'n.n1m.in/41tiup', 'https://t.co/hukvau4hrp', 'http://gs.im/n', 'http://mor.at/qlq/ad8udhi', 'com.preview-domain.com', 'http://www.hkj.buzz/65slwiav', 'HK42753.info', 'http://bit.ly/2isSKeE', 'www.gebyarshopee2019.tk', 'm9s.in/hut3q1ct', 'claro-e.com/4yksm', 'https://aceessink.com/eEHLagt', 'www.bankislam.com', 'https://y.uber.com/1охКwl84', 'www.tim.com.br/applemusic/eesiarls', 'https://docs.google.com/forms/d/e/1FAIlpQLScXa5h', 'bharatpe.in/bharatswipe', 'rml.lu/NPA9NZ', 'https://royalmail.co/schedule-delivery/?GV670517065GB', 'https://bit.ly/2txk2yc?=santander', 'whois.nic.top', 'bitlyk.com/cac11w397', 'http://bit.ly/33ZUxP', 'westpac.account-access.in/', 'h

In [26]:
unique_spamhunter_url = [i for i in unique_spamhunter_url if i!='']

In [27]:
spamhunter_dataset = pd.DataFrame()
spamhunter_dataset['Unique Url'] = unique_spamhunter_url

In [28]:
import tldextract

def FQDN(Url):
    
    url_extract_res = tldextract.extract(Url)
    fqdn = ''
    if url_extract_res.subdomain:
        fqdn = url_extract_res.subdomain + '.' + url_extract_res.domain + '.' + url_extract_res.suffix
        # fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    else:
        fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    
    return fqdn

In [29]:
spamhunter_dataset['FQDN'] = [FQDN(i) for i in spamhunter_dataset['Unique Url']]

len(set(spamhunter_dataset['FQDN']))

8615

In [30]:
a = soupFromUrl('https://facebook.com')

print(a)

[75963, 645, 200, 0]


The below query last ran on 12 Feb 2025

In [31]:
# website_size, text_content_length, status_code, parked = [],[],[],[]

# for i in spamhunter_dataset['FQDN']:
#     a = soupFromUrl('https://'+i)

#     website_size.append(a[0])
#     text_content_length.append(a[1])
#     status_code.append(a[2])
#     parked.append(a[3])


# # parked = [0]*len(spamhunter_dataset)

# # for idx,i in enumerate(spamhunter_dataset['FQDN']):
# #     if spamhunter_dataset['Status Code'][idx]==200:
# #         a = soupFromUrl('https://'+i)
# #         parked[idx] = a[3]
# #         # break

# # print(parked)

C:\Users\mmia43\AppData\Local\Temp\ipykernel_16376\3359563719.py:11: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = bs(req.text,'html')


In [43]:
spamhunter_dataset = pd.read_csv('../Dataset/URL Data/spamhunter Websites Analysis.csv')

In [32]:
# spamhunter_dataset['Website Size in KB'] = website_size
# spamhunter_dataset['Website Textual Content Length'] = text_content_length
# spamhunter_dataset['Status Code'] = status_code

# spamhunter_dataset['Parked'] = parked

In [44]:
for i in spamhunter_dataset:
    print(len(spamhunter_dataset[i]))

15531
15531
15531
15531
15531
15531


In [45]:
spamhunter_dataset

,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,http://www.kycpaytm.com/in.php,www.kycpaytm.com,0,0,-1,0
1,points.co.in,points.co.in,0,0,200,0
2,bit.ly/im3,bit.ly,126176,8761,200,1
3,https://bit.ly/3goHo9H,bit.ly,126176,8761,200,1
4,privee.com,privee.com,114,0,200,1
...,...,...,...,...,...,...
15526,https://apple.com.submit-userdata.services/,apple.com.submit-userdata.services,0,0,-1,0
15527,https://bbva.gestlon-online-es.ru/,bbva.gestlon-online-es.ru,0,0,-1,0
15528,www.bristolrhythms.com,www.bristolrhythms.com,0,0,-1,0
15529,http://www.sccb.inth.icu/,www.sccb.inth.icu,0,0,-1,0


In [46]:
# spamhunter_dataset = pd.read_csv('../Dataset/URL Data/spamhunter Websites Analysis.csv')
spamhunter_dataset = pd.DataFrame.from_dict(spamhunter_dataset)
spamhunter_dataset.head()

,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,http://www.kycpaytm.com/in.php,www.kycpaytm.com,0,0,-1,0
1,points.co.in,points.co.in,0,0,200,0
2,bit.ly/im3,bit.ly,126176,8761,200,1
3,https://bit.ly/3goHo9H,bit.ly,126176,8761,200,1
4,privee.com,privee.com,114,0,200,1


In [36]:
spamhunter_dataset['Status Code'].value_counts()

Status Code
-1      7646
 200    6808
 404     478
 403     457
 400      88
 500      23
 503      12
 501       3
 502       3
 406       2
 410       2
 402       2
 526       1
 429       1
 405       1
 525       1
 434       1
 401       1
 204       1
Name: count, dtype: int64

In [37]:
numbers_to_replace = [501,403, 401]

# Value to replace with
new_value = 200

# Update the column
spamhunter_dataset.loc[spamhunter_dataset['Status Code'].isin(numbers_to_replace), 'Status Code'] = new_value

In [38]:
spamhunter_dataset['Status Code'].value_counts()

Status Code
-1      7646
 200    7269
 404     478
 400      88
 500      23
 503      12
 502       3
 406       2
 410       2
 402       2
 526       1
 429       1
 405       1
 525       1
 434       1
 204       1
Name: count, dtype: int64

In [39]:
spamhunter_dataset['Parked'].value_counts()

Parked
0    11479
1     4052
Name: count, dtype: int64

In [40]:
spamhunter_dataset.head()

,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,http://www.kycpaytm.com/in.php,www.kycpaytm.com,0,0,-1,0
1,points.co.in,points.co.in,0,0,200,0
2,bit.ly/im3,bit.ly,126176,8761,200,1
3,https://bit.ly/3goHo9H,bit.ly,126176,8761,200,1
4,privee.com,privee.com,114,0,200,1


In [41]:
# spamhunter_dataset.to_csv('../Dataset/URL Data/spamhunter Websites Analysis.csv', index=None)

In [42]:
spamhunter_dataset.drop_duplicates(subset='FQDN', inplace=True)
print(len(spamhunter_dataset))
print(spamhunter_dataset['Status Code'].value_counts())
print(spamhunter_dataset['Parked'].value_counts())

8615
Status Code
-1      6528
 200    1665
 404     337
 400      45
 500      19
 503       8
 502       3
 402       2
 406       1
 526       1
 429       1
 410       1
 405       1
 525       1
 434       1
 204       1
Name: count, dtype: int64
Parked
0    8169
1     446
Name: count, dtype: int64
